# Fine-tune YOLO11n — `qr_det` v1 **from datamatrix `best.pt`**

**Zip:** `qr_det.v1i.yolov11.zip` (~225 MB)  
**Base weights:** your Colab-trained datamatrix locator  
`runs/barcode/yolo11n_960_datamatrix_fresh/weights/best.pt`

| Item | Value |
|------|-------|
| Images | **1545** (all in `train/` — auto **90/10** val) |
| Zip classes | `15`, `QR`, `atypical`, `bar_code`, `datamatrix`, `qr_code`, `qrcode`, `tarek` |
| Strategy | **1-class remap** → `0: barcode` (keep all boxes; drop empty-only junk) |
| Start model | **`best.pt`** (not `yolo11n.pt`) |
| imgsz / epochs / batch | **960** / **50** / **16** |
| Run name | `yolo11n_960_qr_det_from_dm` |

**Runtime → Change runtime type → T4 GPU**

### Why 1-class?
- Existing `best.pt` is `{0: datamatrix}` (nc=1) → head stays compatible
- App uses YOLO as **locator** only; ZXing decides QR / DM / 1D
- Zip has messy aliases (`QR`/`qr_code`/`qrcode`) and junk name `15` (583 boxes) — remapping keeps data

### Upload both
1. `qr_det.v1i.yolov11.zip`
2. `best.pt` (datamatrix run)


## 1. GPU + install


In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


In [ ]:
!pip -q install -U ultralytics pyyaml
from ultralytics import YOLO
import ultralytics
print("ultralytics", ultralytics.__version__)


## 2. Upload dataset zip + `best.pt`

- Dataset: **`qr_det.v1i.yolov11.zip`** (`~/Downloads/…`)
- Weights: **`best.pt`** from `yolo11n_960_datamatrix_fresh`


In [ ]:
from google.colab import files
from pathlib import Path

ZIP_PATH = Path("/content/qr_det.v1i.yolov11.zip")
BEST_PT = Path("/content/best.pt")
DATA_ROOT = Path("/content/datasets/qr_det_v1")

if not ZIP_PATH.exists():
    print("Select qr_det.v1i.yolov11.zip …")
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            ZIP_PATH.write_bytes(uploaded[name])
            print("Saved", ZIP_PATH, ZIP_PATH.stat().st_size // (1024 * 1024), "MB")
            break

if not BEST_PT.exists():
    print("Select datamatrix best.pt …")
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".pt"):
            BEST_PT.write_bytes(uploaded[name])
            print("Saved", BEST_PT, BEST_PT.stat().st_size // (1024 * 1024), "MB")
            break

print("ZIP:", ZIP_PATH.exists(), "BEST_PT:", BEST_PT.exists())


In [ ]:
# Optional: load from Google Drive
# from google.colab import drive
# from pathlib import Path
# drive.mount("/content/drive")
# ZIP_PATH = Path("/content/drive/MyDrive/qr_det.v1i.yolov11.zip")
# BEST_PT = Path("/content/drive/MyDrive/yolo11n_960_datamatrix_fresh/best.pt")
# print(ZIP_PATH.exists(), BEST_PT.exists())


## 3. Unpack, remap to 1 class, create val split


In [ ]:
import zipfile
import random
import shutil
import collections
from pathlib import Path
import yaml

ZIP_PATH = Path("/content/qr_det.v1i.yolov11.zip")
DATASETS = Path("/content/datasets")
DATA_ROOT = DATASETS / "qr_det_v1"
VAL_RATIO = 0.10
SEED = 42

# Zip names (index order from data.yaml):
# 0:15  1:QR  2:atypical  3:bar_code  4:datamatrix  5:qr_code  6:qrcode  7:tarek
DROP_IDS = {2, 7}  # atypical, tarek (almost empty / junk)
# Keep 0 (15), 1/5/6 (QR aliases), 3 (bar_code), 4 (datamatrix) → all become class 0

assert ZIP_PATH.exists(), "Upload zip first"

if not (DATA_ROOT / "train" / "images").exists():
    print("Unpacking…")
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_ROOT)

# Roboflow layout: train/images + train/labels at DATA_ROOT
src_img = DATA_ROOT / "train" / "images"
src_lbl = DATA_ROOT / "train" / "labels"
if not src_img.exists():
    # nested folder
    nested = list(DATA_ROOT.rglob("train/images"))
    assert nested, "train/images not found"
    src_img = nested[0]
    src_lbl = nested[0].parent / "labels"
    DATA_ROOT = nested[0].parent.parent

assert src_img.exists() and src_lbl.exists(), (src_img, src_lbl)

# Remap labels in place under a working tree
work = DATA_ROOT / "work_1class"
work_img = work / "images"
work_lbl = work / "labels"
if work.exists():
    shutil.rmtree(work)
work_img.mkdir(parents=True)
work_lbl.mkdir(parents=True)

kept_boxes = 0
dropped_boxes = 0
empty_files = 0
hist_before = collections.Counter()
hist_after = collections.Counter()

for img in sorted(src_img.iterdir()):
    if img.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
        continue
    lbl = src_lbl / (img.stem + ".txt")
    out_lines = []
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            cid = int(float(parts[0]))
            hist_before[cid] += 1
            if cid in DROP_IDS:
                dropped_boxes += 1
                continue
            parts[0] = "0"
            out_lines.append(" ".join(parts))
            kept_boxes += 1
            hist_after[0] += 1
    if not out_lines:
        empty_files += 1
        continue  # skip images with no usable boxes
    shutil.copy2(img, work_img / img.name)
    (work_lbl / (img.stem + ".txt")).write_text("\n".join(out_lines) + "\n")

print("Before class hist:", dict(sorted(hist_before.items())))
print("After class hist:", dict(sorted(hist_after.items())))
print(f"kept_boxes={kept_boxes} dropped_boxes={dropped_boxes} empty_skipped={empty_files}")
print("usable images:", len(list(work_img.iterdir())))

# 90/10 split → Ultralytics layout images/train|val + labels/train|val
final = DATA_ROOT / "yolo"
if final.exists():
    shutil.rmtree(final)
for split in ("train", "val"):
    (final / "images" / split).mkdir(parents=True)
    (final / "labels" / split).mkdir(parents=True)

stems = [p.stem for p in work_img.iterdir()]
random.Random(SEED).shuffle(stems)
n_val = max(1, int(len(stems) * VAL_RATIO))
val_set = set(stems[:n_val])
train_set = set(stems[n_val:])

for stem in stems:
    split = "val" if stem in val_set else "train"
    img = next(work_img.glob(stem + ".*"))
    shutil.copy2(img, final / "images" / split / img.name)
    shutil.copy2(work_lbl / (stem + ".txt"), final / "labels" / split / (stem + ".txt"))

yaml_path = final / "data.yaml"
yaml_path.write_text(
    yaml.dump(
        {
            "path": str(final),
            "train": "images/train",
            "val": "images/val",
            "names": {0: "barcode"},
        },
        sort_keys=False,
    )
)
print(yaml_path.read_text())
print(f"train={len(train_set)} val={len(val_set)}")
Path("/content/DATA_YAML.txt").write_text(str(yaml_path))
Path("/content/BEST_PT.txt").write_text("/content/best.pt")
print("OK — ready to fine-tune")


## 4. Fine-tune from datamatrix `best.pt`


In [ ]:
from ultralytics import YOLO
from pathlib import Path

BEST_PT = Path(Path("/content/BEST_PT.txt").read_text().strip())
DATA_YAML = Path(Path("/content/DATA_YAML.txt").read_text().strip())
assert BEST_PT.exists(), BEST_PT
assert DATA_YAML.exists(), DATA_YAML

PROJECT = "/content/runs/barcode"
RUN_NAME = "yolo11n_960_qr_det_from_dm"
IMGSZ = 960
EPOCHS = 50  # fine-tune; raise to 80–100 if val still climbing
BATCH = 16   # drop to 8 if OOM

model = YOLO(str(BEST_PT))
results = model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    device=0,
)
print("Done. best.pt under", Path(PROJECT) / RUN_NAME / "weights")


## 5. Validate


In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = Path("/content/runs/barcode")
candidates = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert candidates, "No best.pt — did training finish?"
BEST = candidates[0]
print("Using", BEST)

DATA_YAML = Path(Path("/content/DATA_YAML.txt").read_text().strip())
model = YOLO(str(BEST))
metrics = model.val(data=str(DATA_YAML), imgsz=960)
print("mAP50:", float(metrics.box.map50))
print("mAP50-95:", float(metrics.box.map))
print("Precision:", float(metrics.box.mp))
print("Recall:", float(metrics.box.mr))


## 6. Export ONNX (imgsz=960 for scansdk)


In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = Path("/content/runs/barcode")
BEST = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)[0]
print("Exporting", BEST)

model = YOLO(str(BEST))
out = model.export(format="onnx", imgsz=960, simplify=True)
print("ONNX:", out)

onnx_src = Path(out)
onnx_dst = Path("/content/barcode-yolo11n.onnx")
onnx_dst.write_bytes(onnx_src.read_bytes())
print("Wrote", onnx_dst, onnx_dst.stat().st_size // 1024, "KB")


## 7. Download weights

After download, locally:

```bash
cp ~/Downloads/best.pt runs/barcode/yolo11n_960_qr_det_from_dm/weights/
cp ~/Downloads/barcode-yolo11n.onnx public/models/barcode-yolo11n.onnx
```

Then hard-refresh the app and run `npm run test:locate` / `test:barcode` / `test:pharma-multi`.


In [ ]:
from google.colab import files
from pathlib import Path

PROJECT = Path("/content/runs/barcode")
BEST = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)[0]
ONNX = Path("/content/barcode-yolo11n.onnx")
if not ONNX.exists():
    siblings = list(BEST.parent.glob("*.onnx"))
    assert siblings, "Run export cell first"
    ONNX = siblings[0]

print("Downloading", BEST)
files.download(str(BEST))
print("Downloading", ONNX)
files.download(str(ONNX))


## Notes

- **OOM:** set `BATCH = 8`.
- **Runtime restart:** re-upload zip + `best.pt`, or save `/content/runs/barcode` to Drive.
- **Optional 3-class:** if you later want `qr` / `bar_code` / `datamatrix`, remap labels instead of forcing `0`, and start from `yolo11n.pt` (or accept head reset from this `best.pt`).
- Dropping `15` would remove ~583 boxes — we **keep** them as class `barcode` for locator training.
